In [1]:
#!pip install ipywidgets

In [74]:
from ingest import load_faq_data
documents = load_faq_data()

In [75]:
documents[10]

{'id': '316180784f',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.'}

In [76]:
type(documents)

list

In [77]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

113

In [78]:
documents = documents_llm

In [79]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [80]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [81]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [82]:
from dotenv import load_dotenv
from openai import OpenAI
import os

load_dotenv()

# use this for chatgpt
openai_client=OpenAI()
model="gpt-5.4-mini"


openrouter_client = OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
)



model="openai/gpt-oss-120b:free"
model='nvidia/nemotron-3-super-120b-a12b:free'

In [83]:
import json
user_prompt = json.dumps(doc)

In [84]:
user_prompt

'{"id": "74eb249bbf", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "I just discovered the course. Can I still join?", "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."}'

In [85]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [86]:
response = openrouter_client.responses.parse(
    model=model,
    input=messages,
    text_format=Questions
)

In [87]:
response

ParsedResponse[TypeVar](id='gen-1784027602-hxJOaf2UMqh6fECx2mqP', created_at=1784027602.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='nvidia/nemotron-3-super-120b-a12b:free', object='response', output=[ResponseReasoningItem(id='rs_tmp_chfpt903817', summary=[], type='reasoning', content=[Content(text='We need to output 5 questions a student might ask based on this FAQ record. The answer is: Yes, but if you can submit project while still accepting submissions. We need to produce questions that reflect the answer. Use fewer words from the record. The output: list of 5 questions, not too formal, not too short, not too long. Probably something like:\n\n- "Can I still enroll in the course even though I just found out about it?"\n- "If I join now, will I still be able to get a certificate?"\n- "Do I need to finish my project by a certain date to get the certificate?"\n- "Is there a deadline for submitting the final project to receive the certificate?"\n- "What

In [88]:
response.output_parsed.questions

["Can I still sign up for the course after it's started?\n\nIf I join late, will I still be eligible for a certificate?\n\nDo I need to turn in my project by a specific date to earn the certificate?\n\nIs there a cut‑off for submitting the final assignment to receive certification?\n\nWhat if I miss the submission window—can I still get certified?\n\n- Can I still enroll in the course even though I just discovered it?\n- If I start now, will I be able to get a certificate?\n- Do I have to submit my project before a certain deadline to receive the certificate?\n- Is there a final date for handing in the project to qualify for certification?\n- What happens if I don’t submit my project while submissions are still open?\n\n"]

In [89]:
doc

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [90]:
from evaluation_utils import llm_structured

In [91]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['I found this course late — can I still start and join in, or is it too late now?', 'If I join the course after it has already started, will I still be able to get a certificate?', 'Can I enroll in the course anytime, or do I need to sign up before a certain deadline?', 'I just found out about this course. Is it still okay to participate, and what do I need to do if I want the certificate?', 'If I’m joining the course now, when do I have to submit my project to qualify for a certificate?']


In [92]:
usage

ResponseUsage(input_tokens=207, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=128, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=335)

In [93]:
from evaluation_utils import calc_price

In [94]:
calc_price(usage)

{'input_cost': 0.00015525, 'output_cost': 0.000576, 'total_cost': 0.00073125}

In [97]:
records = []

for q in result.questions:
    records.append({
        "question": q,
       "document": doc["id"]
    })

records

[{'question': 'I found this course late — can I still start and join in, or is it too late now?',
  'document': '74eb249bbf'},
 {'question': 'If I join the course after it has already started, will I still be able to get a certificate?',
  'document': '74eb249bbf'},
 {'question': 'Can I enroll in the course anytime, or do I need to sign up before a certain deadline?',
  'document': '74eb249bbf'},
 {'question': 'I just found out about this course. Is it still okay to participate, and what do I need to do if I want the certificate?',
  'document': '74eb249bbf'},
 {'question': 'If I’m joining the course now, when do I have to submit my project to qualify for a certificate?',
  'document': '74eb249bbf'}]

In [26]:
import pandas as pd

In [27]:
pd.DataFrame(records)

,question,document
0,Can I still enroll in the course if I just fou...,74eb249bbf
1,Is it too late to join this course after it ha...,74eb249bbf
2,I just came across the course — can I still pa...,74eb249bbf
3,"If I join the course now, will I still be able...",74eb249bbf
4,What do I need to do to qualify for the certif...,74eb249bbf


In [28]:
from evaluation_utils import llm_structured_retry

In [29]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [30]:
generate_ground_truth(doc)

([{'question': 'Can I still join the course if I just found it, or is it too late now?',
   'document': '74eb249bbf'},
  {'question': 'If I start the course late, will I still be able to get a certificate?',
   'document': '74eb249bbf'},
  {'question': 'Is it okay to join after the course has already started?',
   'document': '74eb249bbf'},
  {'question': 'What do I need to do to qualify for the certificate if I’m joining now?',
   'document': '74eb249bbf'},
  {'question': 'Is there still time to submit the project and receive the certificate?',
   'document': '74eb249bbf'}],
 ResponseUsage(input_tokens=207, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=94, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=301))

In [31]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

100%|██████████| 5/5 [00:09<00:00,  1.84s/it]


In [27]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [28]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

100%|██████████| 113/113 [01:24<00:00,  1.34it/s]


In [29]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

565

In [30]:
ground_truth[10]

{'question': 'Where do students actually join the Office Hours or live workshop sessions if the Zoom link isn’t shared publicly?',
 'document': '489dd1c9d9'}

In [31]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.0876885

In [32]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

0.0876885

In [33]:
df_ground_truth = pd.DataFrame(ground_truth)

In [47]:
df_ground_truth.to_csv("../data/ground_truth.csv", index=False)

In [48]:
len(df_ground_truth)

565